# Colab runner (v2: 7-day plan, LoRA)
Run cells 1-6 at the start of EVERY session (runtime: GPU). Set `GH_USER`, `GH_REPO`, `BRANCH`, name and email in cell 2. Secrets needed: `GITHUB_TOKEN`, `HF_TOKEN`.

In [ ]:
# 1. Drive
from google.colab import drive
drive.mount('/content/drive')
import os
PROJECT_DIR = '/content/drive/MyDrive/clip_project'
os.makedirs(PROJECT_DIR, exist_ok=True)
os.environ['PROJECT_DIR'] = PROJECT_DIR

In [ ]:
# 2. Repo
from google.colab import userdata
GH_USER = '9raveen'                    # repo OWNER, same for everyone
GH_REPO = 'Clip-Compositionality-FGAI'
BRANCH  = 'main'
token = userdata.get('GITHUB_TOKEN')
%cd /content
if not os.path.exists(f'/content/{GH_REPO}'):
    !git clone https://{token}@github.com/{GH_USER}/{GH_REPO}.git
%cd /content/{GH_REPO}
!git fetch origin
!git checkout {BRANCH}
!git pull
!git config user.name  "YOUR NAME"
!git config user.email "YOUR_ID+USERNAME@users.noreply.github.com"

In [ ]:
# 3. Dependencies (skip after a plain 'Restart session'; rerun after a full reset)
!pip install -q -r requirements.txt
!python -m spacy download en_core_web_sm -q

In [ ]:
# 4. Hugging Face login
from huggingface_hub import login
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
login(os.environ['HF_TOKEN'])

In [ ]:
# 5. GPU
import torch; print(torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')
!nvidia-smi | head -12

In [ ]:
# 6. Restore data from Drive to local disk (fast reads)
os.makedirs('data', exist_ok=True)
!cp {PROJECT_DIR}/data/*.jsonl {PROJECT_DIR}/data/splits.json data/ 2>/dev/null || echo 'no data on Drive yet'
if os.path.exists(f'{PROJECT_DIR}/data/coco_subset.zip') and not os.path.exists('data/coco_subset'):
    !unzip -q {PROJECT_DIR}/data/coco_subset.zip -d data/
!ls data

## DAY 1: verify LoRA on the real pretrained model, build data, zero-shot baseline

In [ ]:
!python -m scripts.check_lora --pretrained

In [ ]:
# once: ~8k images, 3 disjoint image-level splits. Add --num_images 25000 to expand later
!python scripts/prepare_coco_subset.py --num_images 8000
!mkdir -p {PROJECT_DIR}/data
!cp data/coco_subset.zip data/captions_*.jsonl data/splits.json {PROJECT_DIR}/data/

In [ ]:
# zero-shot: Winoground + retrieval on the held-out COCO test split
!python -m src.evaluate --config configs/zeroshot.yaml --out_dir {PROJECT_DIR}

## DAY 2: negatives, validation, audit, smoke test

In [ ]:
!python -m src.negatives --captions data/captions_train.jsonl --out data/negatives_train.jsonl
!python -m src.negatives --captions data/captions_val.jsonl   --out data/negatives_val.jsonl
!cp data/negatives_*.jsonl data/negatives_*_stats.json {PROJECT_DIR}/data/

In [ ]:
# audit sample: 40 per type -> label in a spreadsheet, then score
!python -m scripts.audit_negatives sample --negatives data/negatives_train.jsonl --n 40 --out {PROJECT_DIR}/audit.csv
# after labelling (column `label`: 1 = clearly false caption, 0 = could be true / ambiguous):
# !python -m scripts.audit_negatives score --csv {PROJECT_DIR}/audit.csv

In [ ]:
# smoke test: 5 steps, prints img/s. Use it to budget every later run.
!python -m src.train --config configs/lora_hn.yaml --seed 0 --out_dir /content/smoke --max_steps 5

## DAYS 3-5: experiments (resumable: rerun the same cell after a disconnect)

In [ ]:
!python scripts/run_all.py --tier main

In [ ]:
!python scripts/run_all.py --tier ablation

In [ ]:
# optional, only if time remains
# !python scripts/run_all.py --tier optional

## Save small results to GitHub

In [ ]:
!mkdir -p results
!cp {PROJECT_DIR}/results/*.json results/ 2>/dev/null
!git add results/*.json data/splits.json
!git commit -m "add results"
!git push origin {BRANCH}